# 1. 타입
- 파이썬은 동적 타입 언어(dynamic typing language)
- 변수 선언 시 타입을 명시하지 않아도 되고, 값이 대입될 때 자동으로 타입이 결정

## 1.1 type과 type()
- type()은 객체의 타입을 확인할 때 사용하는 함수. 인자의 타입을 type 객체로 반환
- 인자를 type 객체로 변환하는 것으로 볼 수도 있다.
- 사실 type은 클래스이고 type()은 생성자이다.

In [38]:
class A : pass

a = A()
type(a)  # __main__.A   메인 모듈의 A 클래스의 객체를 반환

__main__.A

In [11]:
print(type(10))       # <class 'int'>
print(type("hi"))     # <class 'str'>
print(type([1,2,3]))  # <class 'list'>
print(type(None))     # <class 'NoneType'>
print(type(type(10))) # <class 'type'>

<class 'int'>
<class 'str'>
<class 'list'>
<class 'NoneType'>
<class 'type'>


In [31]:
dir(type)    # callable.  __call__을 가지고 있다. type()이 가능

['__abstractmethods__',
 '__annotations__',
 '__base__',
 '__bases__',
 '__basicsize__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dictoffset__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__flags__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__instancecheck__',
 '__itemsize__',
 '__le__',
 '__lt__',
 '__module__',
 '__mro__',
 '__name__',
 '__ne__',
 '__new__',
 '__or__',
 '__prepare__',
 '__qualname__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__ror__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasscheck__',
 '__subclasses__',
 '__subclasshook__',
 '__text_signature__',
 '__type_params__',
 '__weakrefoffset__',
 'mro']

In [15]:
len(dir(type))

48

In [37]:
print(type.__base__)   # object. type의 조상은 object. 모든 클래스의 조상은 object
print(object.__base__) # None. 최고 조상이므로 조상(__base__)이 없다.
print(issubclass(type, object)) # type은 object의 자손
print(isinstance(type, object)) # type도 객체(타입 객체)
print(isinstance(object, type)) # object클래스도 타입

print(issubclass(int, type))   # int와 type은 상속관게x
print(issubclass(int, object)) # int는 object의 자손 타입


<class 'object'>
None
True
True
True
False
True


In [22]:
type(object)  # object도 타입(클래스)

type

# 2. 메타 클래스(metaclass)

## 2.1 type
- type은 기본 메타 클래스
- 기본적으로 모든 클래스는 type으로 만들어진다(다른 클래스로 변경도 가능)


In [42]:
class A:       # class A(metaclass=type): 이 생략된 형태
    pass

type(A) is type      # True


True

In [45]:
class MyMeta(type):
    pass

class A(metaclass=MyMeta):
    pass

type(A) is MyMeta  # True
# type(A) is type  # False


True

### 동적으로 클래스 생성하기
type(name, bases, dict)

In [40]:
# class A:
#     x = 10

A = type("A", (), {"x": 10})  # 위와 동일


## 2.1 사용자 정의 메타 클래스 
- 기본 메타 클래스를 상속을 받아 사용자 정의 메타 클래스를 만들 수 있다.

In [ ]:
class mytype(type):
    def __call__(cls, *args):
        print("mytype __call__()")
        
class A(metaclass=mytype): # 메타클래스를 mytype으로 지정
    pass

a = A()
print(a.__class__)  # NoneType  보통은 __main__.A라고 나와야함
print(A.__class__)  # __main__.mytype

mytype __call__()
<class 'NoneType'>
<class '__main__.mytype'>


In [71]:
class mytype(type):
    def __call__(cls, *args):  # __call__이 __new__, __init__을 호출해야 함.
        print("mytype __call__()")
        instance = cls.__new__(cls)
        cls.__init__(instance, *args)
        return instance        

class A(metaclass=mytype): # 메타클래스를 mytype으로 지정
    pass

a = A()
print(a.__class__)  # __main__.A  이제 제대로 나옴
print(A.__class__)  # __main__.mytype

mytype __call__()
<class '__main__.A'>
<class '__main__.mytype'>


### 메타 클래스로 싱글턴 구현하기
-   메타클래스 정의할 때 클래스가 호출되는 '__call__' 메소드 내에 
-  사용자 클래스에 _instance 속성이 있으면 사용자 클래스에 저장된 인스턴스를 반환

In [ ]:
class singleton_meta(type):
    def __call__(self, *args):
        print("__call__ in type")
        if hasattr(self, '_instance'):
            return self._instance
        
        instance = self.__new__(self)    # 객체를 생성해서 인스턴스 변수로 추가
        self.__init__(instance, *args)
        return instance

In [84]:
class Singleton(metaclass=singleton_meta):
    def __call__(self):
        pass  # __new__, __init__을 호출하지 않음. 메타 클래스의 __call__이 호출됨
    
    def __new__(cls):
        print("__new__ in A")
        instance = super().__new__(cls) 
        cls._instance = instance
        return instance
    
    def __init__(self, x, y, z):
        print('__init__ in A')
        self.x = x
        self.y = y
        self.z = z
        
s = Singleton(1,2,3)

__call__ in type
__new__ in A
__init__ in A


In [82]:
s2 = Singleton(4,5,6) # 항상 같은 객체를 반환
s is s2   # True

__call__ in type


True

### 클래스 자동 등록 (Registry 패턴)
- 플러그인 시스템
- ORM 모델 자동 등록
- 커맨드 패턴 자동 수집  
👉 Django ORM이 이런 구조를 사용

In [85]:
class RegistryMeta(type):
    registry = {}

    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        if name != "Base":
            mcls.registry[name] = cls
        return cls


class Base(metaclass=RegistryMeta):
    pass


class A(Base):
    pass


class B(Base):
    pass


print(RegistryMeta.registry)
# {'A': <class '__main__.A'>, 'B': <class '__main__.B'>}


{'A': <class '__main__.A'>, 'B': <class '__main__.B'>}


### 클래스 생성 시 규칙 강제하기
- 특정 메서드 구현을 강제할 수 있습니다.
- 프레임워크 API 강제
- DSL 설계
- 특정 인터페이스 강제(collections.abc보다 더 강력하게 제어 가능)

In [87]:
class RequireMethodMeta(type):
    def __new__(mcls, name, bases, namespace):
        if "run" not in namespace:
            raise TypeError(f"{name} must define run() method")
        return super().__new__(mcls, name, bases, namespace)


class Job(metaclass=RequireMethodMeta):
    def run(self):
        pass  # OK


class BadJob(metaclass=RequireMethodMeta):
    pass    # TypeError 발생. run()을 구현하지 않았음.


TypeError: BadJob must define run() method

### 클래스 속성 자동 변환 (ORM 스타일)
- 클래스 정의를 읽어서 자동으로 처리합
- Django ORM
- SQLAlchemy
- Pydantic
- FastAPI 내부 모델 처리

In [88]:
class Field:
    def __init__(self, type_):
        self.type_ = type_


class ModelMeta(type):
    def __new__(mcls, name, bases, namespace):
        fields = {}
        for k, v in namespace.items():
            if isinstance(v, Field):
                fields[k] = v

        namespace["_fields"] = fields
        return super().__new__(mcls, name, bases, namespace)


class User(metaclass=ModelMeta):
    name = Field(str)
    age = Field(int)


print(User._fields)
# {'name': Field(str), 'age': Field(int)}

{'name': <__main__.Field object at 0x1065b8ce0>, 'age': <__main__.Field object at 0x1065b9cd0>}


## 2.2 상속 vs. 메타 클래스

- 메타클래스는 클래스를 생성하고 동작을 제어하는 클래스. 제어가 목적
- 상속은 다른 클래스의 속성(변수)과 메서드(함수)를 물려받는 기능

| 항목      | **상속 (Inheritance)** | **메타 클래스 (Metaclass)** |
| ------- | -------------------- | ---------------------- |
| 대상      | 클래스 → 서브클래스          | 메타클래스 → 클래스            |
| 목적      | 코드 재사용, 행위 확장        | 클래스 생성/행위 제어           |
| 관계      | 클래스 간 계층 구조          | 클래스 생성 메커니즘 자체         |
| 동작 시점   | 런타임 인스턴스 행동          | 클래스 정의 시점              |
| 사용하는 경우 | 일반 OOP 설계            | 프레임워크, DSL, 반복 규칙      |
